In [1]:
from nltk.corpus import stopwords

stopwords.words('english')

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

In [2]:
from sqlalchemy import create_engine
import pandas as pd
import re
import string
import nltk
pd.set_option('display.max_colwidth', 100)

In [3]:
QUERY=""" 
SELECT
trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
,category_name "label"
FROM oneflare_reports.business_master bm
WHERE bm.total_quotes_made >= 3 and category_name is not null
"""

In [4]:
def make_engine():
    user = 'xxx'
    pw = 'xxx'
    db_name = 'xxx'
    host = 'xxx.xx.xx'
    port = 5439
    engine_string = "postgresql://%s:%s@%s:%s/%s" % (
        user, pw, host, port, db_name)
    engine = create_engine(engine_string)
    return engine

redshift = make_engine()
data=pd.read_sql(QUERY,redshift)


In [5]:
data.head()

,feature,label
0,"Anaxim Accounting Solutions The company is a qualified, professional accountant who deals with a...",Accountant
1,RCS Websites Let RCS Websites breath life in to your existing website!\r\nOur websites are simpl...,SEO and SEM
2,Intouch Carpet Cleaning Services InTouch Carpet and Pest Control is a family owned business who ...,Pest Control
3,Shannon Thomas Carpenter Licensed Carpenter\r\n - Small Decks and Pergolas as well as re deck...,Carpenter
4,"M&M Cabinets Custom-made Cabinets for kitchen, laundries, bathroom and anywhere in your house. 3...",Builder


In [6]:
stopwords = nltk.corpus.stopwords.words('english')
wn = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [79]:
'123'.isdigit()

True

In [7]:
def clean_text_by_porter_stem(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens if (word not in stopwords) and (word.isdigit() ==False) ]
    
    return text

def clean_text_by_wordnet_lemmatize(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [wn.lemmatize(word)  for word in tokens if (word not in stopwords) and (word.isdigit() ==False)] 
    
    return text

In [13]:
#data.drop('feature_cleaned', axis=1, inplace=True)

In [28]:
data['feature_cleaned_ps']=data['feature'].apply(lambda x: clean_text_by_porter_stem( ('' if x is None else x).lower()))
data['feature_cleaned_wd']=data['feature'].apply(lambda x: clean_text_by_wordnet_lemmatize( ('' if x is None else x).lower()))

In [29]:
data.head()

,feature,label,feature_cleaned_ps,feature_cleaned_wd
0,"Anaxim Accounting Solutions The company is a qualified, professional accountant who deals with a...",Accountant,"[anaxim, account, solut, compani, qualifi, profession, account, deal, array, account, issu, prov...","[anaxim, accounting, solution, company, qualified, professional, accountant, deal, array, accoun..."
1,RCS Websites Let RCS Websites breath life in to your existing website!\r\nOur websites are simpl...,SEO and SEM,"[rc, websit, let, rc, websit, breath, life, exist, websit, websit, simpl, clean, profession, sur...","[rcs, website, let, rcs, website, breath, life, existing, website, website, simple, clean, profe..."
2,Intouch Carpet Cleaning Services InTouch Carpet and Pest Control is a family owned business who ...,Pest Control,"[intouch, carpet, clean, servic, intouch, carpet, pest, control, famili, own, busi, valu, old, f...","[intouch, carpet, cleaning, service, intouch, carpet, pest, control, family, owned, business, va..."
3,Shannon Thomas Carpenter Licensed Carpenter\r\n - Small Decks and Pergolas as well as re deck...,Carpenter,"[shannon, thoma, carpent, licens, carpent, small, deck, pergola, well, deck, repair, attent, det...","[shannon, thomas, carpenter, licensed, carpenter, small, deck, pergola, well, decking, repair, a..."
4,"M&M Cabinets Custom-made Cabinets for kitchen, laundries, bathroom and anywhere in your house. 3...",Builder,"[mm, cabinet, custommad, cabinet, kitchen, laundri, bathroom, anywher, hous, 3d, comput, draw, a...","[mm, cabinet, custommade, cabinet, kitchen, laundry, bathroom, anywhere, house, 3d, computer, dr..."


In [31]:
print(type(data.iloc[0]['feature_cleaned_ps']))
print(type(data.iloc[0]['feature_cleaned_wd']))

<class 'list'>
<class 'list'>


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [9]:
tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
tfidf_vectorized_matrix_ps = tfidf_vectorizer_ps.fit_transform(data['feature'] )



In [10]:
tfidf_vectorized_matrix_ps.shape

(38897, 48583)

In [21]:
tfidf_vectorized_df_ps = pd.DataFrame(tfidf_vectorized_matrix_ps.toarray())

MemoryError: 

In [18]:
tfidf_vectorized_df_ps.head()

,0,1,2,3,4,5,6,7,8,9,...,48564,48565,48566,48567,48568,48569,48570,48571,48572,48573
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [62]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])

In [63]:
tfidf_vectorized_matrix_wn.shape

(39014, 58815)

In [ ]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])
tfidf_vectorized_df_wn = pd.DataFrame(tfidf_vectorized_matrix_wn.toarray())

In [19]:
tfidf_vectorized_df_ps_h = tfidf_vectorized_df_ps
tfidf_vectorized_df_ps_h.columns = tfidf_vectorizer_ps.get_feature_names()
tfidf_vectorized_df_ps_h.head()

,,00,000107l,000684l,000710l,001023l,01000sqm,0118651a01,012km,024481i,...,ўвўfrighten,ўвўhave,ўвўin,ўёўў,ўўв,中國理解,在墨尔本的中国婚礼乐队,ﬁnanc,ﬁnancial,ﬁngertip
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
tfidf_vectorized_df_wn_h = tfidf_vectorized_df_wn
tfidf_vectorized_df_wn_h.columns = tfidf_vectorizer_wn.get_feature_names()
tfidf_vectorized_df_wn_h.head()

In [11]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support as score
import time

In [65]:
print(dir(RandomForestClassifier))
print(RandomForestClassifier())

['__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_cache', '_abc_negative_cache', '_abc_negative_cache_version', '_abc_registry', '_estimator_type', '_get_param_names', '_make_estimator', '_set_oob_score', '_validate_X_predict', '_validate_estimator', '_validate_y_class_weight', 'apply', 'decision_path', 'feature_importances_', 'fit', 'get_params', 'predict', 'predict_log_proba', 'predict_proba', 'score', 'set_params']
RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
            max_depth=None, max_features='auto', max_leaf_nodes=None,
            min_impurity_decrease=0.0,

In [66]:
print(dir(GradientBoostingClassifier))
print(GradientBoostingClassifier())

['_SUPPORTED_LOSS', '__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_cache', '_abc_negative_cache', '_abc_negative_cache_version', '_abc_registry', '_check_initialized', '_check_params', '_clear_state', '_decision_function', '_estimator_type', '_fit_stage', '_fit_stages', '_get_param_names', '_init_decision_function', '_init_state', '_is_initialized', '_make_estimator', '_resize_state', '_staged_decision_function', '_validate_estimator', '_validate_y', 'apply', 'decision_function', 'feature_importances_', 'fit', 'get_params', 'n_features', 'predict', 'predict_log_proba', 'predict_proba', 'score', 's

## Start to evaluate models

In [12]:
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(tfidf_vectorized_matrix_ps, data['label'], test_size=0.2)

In [28]:
print(X_train)
print(y_train)

  (0, 12554)	0.897560979187
  (0, 43683)	0.418577057734
  (0, 38357)	0.138482978662
  (1, 546)	0.153514794957
  (1, 25221)	0.140580537403
  (1, 27985)	0.614059179828
  (1, 30343)	0.129450761432
  (1, 31123)	0.116775518268
  (1, 10302)	0.0797108439834
  (1, 7831)	0.176736539174
  (1, 43756)	0.0797108439834
  (1, 1885)	0.0838096269878
  (1, 5411)	0.0679026764305
  (1, 31732)	0.089084966057
  (1, 38273)	0.102735537169
  (1, 44046)	0.0800556182295
  (1, 45950)	0.0960768681014
  (1, 17635)	0.240989165277
  (1, 47144)	0.0874223538907
  (1, 14034)	0.0742547705572
  (1, 41046)	0.106732115752
  (1, 32781)	0.138885720703
  (1, 39241)	0.0787243392195
  (1, 24799)	0.064152586744
  (1, 31790)	0.0704076165584
  :	:
  (31109, 4269)	0.0377460871041
  (31109, 35272)	0.0577218965356
  (31109, 24686)	0.0417189988265
  (31109, 21468)	0.0281246360286
  (31109, 3707)	0.100935388312
  (31109, 14571)	0.0423268753826
  (31109, 46080)	0.0482403344871
  (31109, 11701)	0.0361762110722
  (31109, 2750)	0.0289037009

In [31]:
print(X_test)

  (0, 10272)	0.248899690119
  (0, 44883)	0.248899690119
  (0, 3670)	0.2658434338
  (0, 23632)	0.5316868676
  (0, 23979)	0.210811382518
  (0, 32119)	0.192613791953
  (0, 11749)	0.22008787539
  (0, 25430)	0.222044491763
  (0, 19132)	0.130824386178
  (0, 26435)	0.119020203111
  (0, 14463)	0.18765297304
  (0, 29282)	0.11753117282
  (0, 35354)	0.163801977225
  (0, 31704)	0.106775275246
  (0, 38178)	0.142796360154
  (0, 39919)	0.138101859336
  (0, 5235)	0.220350590333
  (0, 12337)	0.0936864965212
  (0, 15249)	0.12099438661
  (0, 45669)	0.102670656056
  (0, 25754)	0.0851553043744
  (0, 14683)	0.0885781407425
  (0, 11701)	0.0911815794986
  (0, 44489)	0.112900334261
  (0, 21584)	0.073057114076
  :	:
  (7776, 14571)	0.0995837462847
  (7776, 17076)	0.0685402501629
  (7776, 37387)	0.105277003827
  (7776, 30447)	0.0601834483687
  (7776, 40154)	0.0636014563273
  (7776, 26337)	0.284373096009
  (7776, 44750)	0.0794525675833
  (7776, 33974)	0.0831898056697
  (7776, 41667)	0.0833808194948
  (7776, 38357

In [15]:
def train_RF(n_est, depth):
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, n_jobs=-1)
    rf_model = rf.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    precision, recall, fscore, support = score(y_test, y_pred)
    print('Est: {} / Depth: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        n_est, depth, precision, recall,
        round((y_pred==y_test).sum() / len(y_pred), 3)))

In [34]:
for n_est in [10, 50, 100]:
    for depth in [10, 20, 30, None]:
        train_RF(n_est, depth)

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 10 / Depth: 10 ---- Precision: [ 0.62993763  0.          0.16666667  0.          0.46153846  0.6         0.
  0.          0.          0.          0.          0.24217586  0.
  0.66666667  0.          0.66666667  0.          0.          0.36842105
  0.72916667  0.          0.83870968  0.85714286  0.12417686  0.61111111
  0.          0.33333333  0.88888889  0.          0.          0.          0.
  0.44444444  0.          0.          0.          0.          0.65384615
  0.          0.          0.          0.41208791  0.          0.71428571
  0.5         0.          0.          0.          0.          0.
  0.14285714  0.          0.          0.          0.          0.
  0.42635659  0.71428571  0.75728155  0.          0.5         0.
  0.32962963  0.33333333  0.5         0.          0.          0.
  0.62987013  0.71014493  0.          1.          0.          0.51456311
  0.          0.          0.421875    0.          0.          0.          0.
  0.          0.          0.61256545  0.   

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)


Est: 10 / Depth: 20 ---- Precision: [ 0.71496437  0.75        0.57281553  0.          0.36666667  0.40625     0.5
  0.          0.          0.          0.82        0.23483226  0.
  0.16666667  0.          0.          0.6969697   0.          0.
  0.54545455  0.72727273  0.25        0.90666667  1.          0.23382273
  0.57608696  0.68674699  0.66666667  0.9         0.          0.54166667
  0.09090909  0.          0.75        1.          0.14285714  0.36        0.
  0.69411765  0.          0.          0.          0.42939481  0.
  0.67647059  0.44        0.          0.23076923  0.375       0.          0.
  0.11538462  1.          0.41935484  0.          1.          0.
  0.42857143  0.6875      0.49107143  0.33333333  0.55555556  0.
  0.33870968  0.25        0.33333333  0.          0.33333333  0.
  0.66847826  0.82653061  0.          0.86046512  0.          0.63461538
  0.          0.09090909  0.59139785  0.          0.          0.33333333
  0.30769231  0.          0.8         0.66197183  

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


...

Est: 100 / Depth: 20 ---- Precision: [ 0.78554779  0.84615385  0.68888889  0.          0.47058824  0.          0.
  0.          0.          0.          0.9047619   0.27054264  0.
  0.71428571  0.          1.          0.          1.          0.66666667
  0.9047619   0.          0.97241379  1.          0.22107969  0.82208589
  0.72077922  0.80851064  0.          0.          0.84285714  0.          0.
  0.66666667  0.          0.33333333  0.35106383  0.          0.81012658
  0.33333333  0.          0.          0.42487047  0.          0.81578947
  0.42105263  0.          0.          0.          0.          0.          0.4
  0.          0.18181818  0.          1.          0.          0.4829932
  0.71428571  0.68103448  0.          0.          0.          0.38487973
  0.          0.          0.          0.          0.          0.74489796
  0.85087719  0.          0.8875      0.          0.66507177  0.          0.
  0.68509615  0.          0.          0.          0.          0.          0.
  

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 100 / Depth: 30 ---- Precision: [ 0.79342723  0.92857143  0.61983471  0.75        0.44444444  0.66666667
  0.5         1.          0.          0.          0.91111111  0.27722008
  0.          0.81818182  0.          0.93939394  0.          0.
  0.69230769  0.88157895  0.          0.95333333  0.83333333  0.33754941
  0.75138122  0.7183908   0.88        0.5         0.          0.79487179
  0.          0.          0.73684211  0.5         0.61538462  0.35310734
  0.          0.75581395  0.54545455  0.          0.          0.42821159
  0.          0.91428571  0.48235294  0.          1.          0.          0.
  0.          0.55555556  0.          0.47826087  0.          0.61904762
  0.          0.47945205  0.7027027   0.70731707  0.          1.          0.
  0.38811189  1.          0.          0.          0.          0.
  0.71014493  0.86842105  0.          0.87058824  0.33333333  0.68215159
  0.          0.          0.68446602  0.          0.          0.          0.4
  0.          1. 

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 100 / Depth: None ---- Precision: [ 0.87532468  0.78947368  0.53571429  0.82352941  0.42857143  0.61643836
  0.7         1.          0.          0.          0.88235294  0.42346939
  0.          0.65517241  0.          0.94117647  0.6         0.18518519
  0.65168539  0.8045977   0.          0.94117647  0.83333333  0.59851852
  0.75824176  0.68556701  0.78947368  0.93333333  0.66666667  0.7721519
  0.18421053  0.28571429  0.77419355  0.5483871   0.5625      0.38138138
  0.66666667  0.73529412  0.43478261  0.          0.          0.48181818
  0.          0.72580645  0.50515464  0.5         0.4         0.55555556
  0.25        0.          0.41666667  1.          0.44859813  0.
  0.54545455  1.          0.50757576  0.7012987   0.70895522  0.33333333
  1.          0.          0.39090909  1.          0.07317073  0.4
  0.54545455  0.          0.85632184  0.87610619  1.          0.87777778
  0.52173913  0.75833333  1.          0.29268293  0.76630435  0.
  0.71428571  0.          0.71428571

In [14]:
from sklearn.model_selection import GridSearchCV

In [38]:
rf = RandomForestClassifier()
param = {'n_estimators': [10, 150, 300],
        'max_depth': [30, 60, 90, None]}


In [ ]:
train_RF(150, None)

In [16]:
train_RF(300, None)

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 300 / Depth: None ---- Precision: [ 0.91269841  0.79166667  0.50714286  0.79310345  0.47058824  0.57971014
  0.76470588  0.71428571  0.          0.25        0.78571429  0.41724942
  0.          0.65957447  0.          0.          0.83050847  1.          0.
  0.68421053  0.82978723  0.          0.91724138  0.73913043  0.55036496
  0.77272727  0.59585492  0.796875    0.86666667  0.          0.77142857
  0.26804124  0.75        0.69565217  0.53571429  0.71794872  0.37864078
  0.          0.79        0.28205128  1.          0.25        0.54754098
  0.          0.8372093   0.50666667  0.5         0.21568627  0.625       0.4
  0.          0.35087719  0.95        0.34193548  0.          0.6         0.9
  0.46022727  0.79310345  0.73451327  1.          0.92857143  0.
  0.41176471  0.75        0.03703704  0.66666667  0.57142857  0.33333333
  0.88        0.80357143  0.75        0.85        0.41666667  0.65970149
  1.          0.11538462  0.76442308  1.          1.          0.
  0.63157895  

In [ ]:
gs = GridSearchCV(rf, param, cv=5, n_jobs=-1)
gs_fit = gs.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(gs_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

In [15]:
def train_GB(est, max_depth, lr):
    gb = GradientBoostingClassifier(n_estimators=est, max_depth=max_depth, learning_rate=lr)
    gb_model = gb.fit(X_train, y_train)
    y_pred = gb_model.predict(X_test)
    precision, recall, fscore, train_support = score(y_test, y_pred)
    print('Est: {} / Depth: {} / LR: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        est, max_depth, lr, precision,recall, 
        round((y_pred==y_test).sum()/len(y_pred), 3)))

In [16]:
from ipywidgets import FloatProgress
from IPython.display import display




In [ ]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [50, 100, 150]:
    for max_depth in [3, 7, 11, 15]:
        print("\n\nTraining combination: n_est:{n_est} - max_depth:{max_depth}".format(n_est=str(n_est),max_depth=str(max_depth)))
        progressBar.value+=1
        count += 1
        train_GB(n_est, max_depth, 0.01)

A Jupyter Widget



Training combination: 50 - 3


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)


Est: 50 / Depth: 3 / LR: 0.01 ---- Precision: [ 0.93452381  0.96        0.73451327  0.75        0.28        0.48        0.71875
  0.45        0.          0.41666667  0.81538462  0.34558824  0.
  0.72972973  0.          0.16666667  0.74545455  0.83333333  0.07692308
  0.61458333  0.83333333  0.          0.90833333  0.75        0.65492958
  0.81761006  0.7         0.69565217  0.76470588  0.5         0.81428571
  0.30508475  0.57142857  0.90243902  0.53125     0.53846154  0.28851541
  0.66666667  0.65137615  0.38461538  0.83333333  1.          0.49832776
  0.          0.86956522  0.43434343  0.66666667  0.19354839  0.66666667
  0.08333333  0.55555556  0.40243902  1.          0.4159292   0.          0.53125
  1.          0.55555556  0.81578947  0.8         0.53846154  0.76470588
  0.          0.36394558  0.84615385  0.11363636  0.75        0.59259259
  0.          0.91156463  0.8877551   0.          0.89041096  0.41176471
  0.68543046  0.625       0.18681319  0.77777778  0.          0.6666

Est: 50 / Depth: 15 / LR: 0.01 ---- Precision: [ 0.94864048  0.95238095  0.62601626  0.59375     0.34285714  0.49019608
  0.64        0.64285714  0.          0.          0.77777778  0.27158099
  0.          0.64516129  0.          0.2         0.84444444  0.77777778
  0.03846154  0.52991453  0.79710145  0.          0.90677966  0.8
  0.66793893  0.82550336  0.69871795  0.70212766  0.92857143  0.
  0.81355932  0.17682927  0.5         0.92105263  0.48275862  0.46666667
  0.31481481  0.42857143  0.6         0.30434783  0.83333333  1.          0.4924812
  0.          0.82978723  0.44615385  0.52941176  0.09174312  0.42307692
  0.25        0.14814815  0.32467532  1.          0.37162162  0.
  0.51612903  1.          0.56521739  0.8358209   0.77777778  0.4
  0.90909091  0.          0.37089202  0.8         0.08823529  0.5
  0.45714286  0.07692308  0.93377483  0.91304348  0.25        0.92647059
  0.44444444  0.67054264  0.71428571  0.15789474  0.78456592  0.          0.6
  0.          0.37209302 

In [ ]:
gb = GradientBoostingClassifier()
param = {
    'n_estimators': [50, 100, 150], 
    'max_depth': [7, 11, 15],
    'learning_rate': [0.01]
}

clf = GridSearchCV(gb, param, cv=5, n_jobs=-1)
cv_fit = clf.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

In [ ]:
rf = RandomForestClassifier(n_estimators=150, max_depth=None, n_jobs=-1)
rf_model = rf.fit(tfidf_vectorized_matrix_ps, data['label'])

In [ ]:
QUERY = """ 
SELECT
bm.id
,trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
FROM oneflare_reports.business_master bm
WHERE bm.total_quotes_made < 3
"""

In [ ]:
data_to_predict=pd.read_sql(QUERY,redshift)

In [ ]:
from sklearn.externals import joblib
joblib.dump(rf_model, 'filename.pkl') 
#clf = joblib.load('filename.pk1')